# Pydantic v2 데이터 검증

문자열 입력을 자동 변환할 때와 엄격한 자료형·범위 제약을 적용할 때 결과가 어떻게 달라지는지 확인합니다.

## 실행 환경 준비

이 노트북은 Pydantic 2.13.5를 사용합니다. B 경로 선택 실습입니다. 03절의 딕셔너리, 05절의 예외, 06절의 클래스, 07절의 타입힌트를 연결합니다. 이 내용은 Ch 18의 API 입력과 Ch 30의 서비스 경계에서 다시 사용합니다.

In [1]:
# 교재와 같은 Pydantic 버전을 현재 커널에 설치합니다.
%pip install -q pydantic==2.13.5

Note: you may need to restart the kernel to use updated packages.


## 기본 실행: 입력을 모델로 바꾸기

`class LooseJob(BaseModel)`은 Pydantic의 검증 기능을 물려받은 새 모델 클래스를 정의합니다. 들여쓴 `retries: int = Field(ge=0, le=5)`는 필드 이름, 예상 자료형, 허용 범위를 선언합니다. `model_validate()`를 호출하는 시점에 딕셔너리 입력을 검사하며, 성공하면 `LooseJob` 객체를 반환합니다. 기본 설정은 변환할 수 있는 문자열을 대상 자료형으로 변환합니다.

In [2]:
# 문자열 입력이 정수로 변환되는 기본 모델을 만듭니다.
from pydantic import BaseModel, Field, ValidationError

class LooseJob(BaseModel):
    retries: int = Field(ge=0, le=5)

job = LooseJob.model_validate({"retries": "3"})
print(job)
print(type(job.retries).__name__)

retries=3
int


## 검증된 모델을 딕셔너리로 바꾸기

`job`은 검증을 통과한 모델 객체입니다. `job.retries`로 정수 필드 값을 읽고, `model_dump()`를 호출하면 검증된 값을 파이썬 딕셔너리로 반환받습니다. 입력의 문자열 `"3"`이 출력 딕셔너리에서는 정수 `3`인지 확인합니다.

In [3]:
# 검증이 끝난 모델을 사전으로 반환해 자료형 변환을 확인합니다.
dumped = job.model_dump()
print(dumped)
assert dumped == {"retries": 3}

{'retries': 3}


## 값 하나를 바꿔 비교: 필드 범위

`Field`의 `ge`는 이상, `le`는 이하 범위를 지정합니다. 입력 5는 검증된 모델 객체를 반환하지만, 입력 8은 모델 객체 대신 `ValidationError`를 발생시킵니다. `errors()`가 반환한 목록에서 첫 오류의 `type`과 `loc`을 읽습니다.

In [4]:
# 허용 범위를 벗어난 정수의 구조화된 오류를 확인합니다.
class BoundedJob(BaseModel):
    retries: int = Field(ge=0, le=5)

valid_bound = BoundedJob.model_validate({"retries": 5})
print("허용 경계:", valid_bound.retries)
assert valid_bound.retries == 5

try:
    BoundedJob.model_validate({"retries": 8})
except ValidationError as error:
    issue = error.errors()[0]
    print(issue["type"], issue["loc"])
    assert issue["type"] == "less_than_equal"
else:
    raise AssertionError("상한 초과 입력을 거부해야 합니다.")


허용 경계: 5
less_than_equal ('retries',)


## Before: 자동 변환이 잘못된 입력 형식을 가림

외부 입력 규칙에서 정수만 허용해야 하는데 기본 변환을 그대로 두면 문자열 `"3"`도 통과합니다.

In [5]:
# 기본 모드가 문자열 재시도 횟수를 허용하는지 확인합니다.
loose = LooseJob.model_validate({"retries": "3"})
print("입력 '3' 통과 ->", loose.retries)
assert loose.retries == 3

입력 '3' 통과 -> 3


## After: 엄격한 필드와 범위 제약 적용

`strict=True`는 문자열을 정수로 자동 변환하지 않습니다. 같은 문자열 딕셔너리를 `model_validate()`에 전달하면 `int_type` 오류가 발생하고, 정수 3을 전달하면 모델 객체가 반환됩니다. 범위 제약도 같은 필드에 함께 선언하여 자료형 규칙과 값 규칙을 한곳에 둡니다.

In [6]:
# 같은 재시도 입력을 엄격한 필드로 검사합니다.
class StrictJob(BaseModel):
    retries: int = Field(strict=True, ge=0, le=5)

try:
    StrictJob.model_validate({"retries": "3"})
except ValidationError as error:
    issue = error.errors()[0]
    print(issue["type"], issue["loc"])
    assert issue["type"] == "int_type"

else:
    raise AssertionError("문자열 입력을 거부해야 합니다.")

valid = StrictJob.model_validate({"retries": 3})
print("정수 3 통과 ->", valid.retries)

int_type ('retries',)
정수 3 통과 -> 3
